# Clase 135 — RNNs: neuronas recurrentes, BPTT

Las **redes recurrentes (RNN)** son la primera arquitectura para **secuencias**:
la **misma celda** se aplica en cada paso temporal y un **estado oculto** `h_t`
acumula el contexto. Se entrenan con **BPTT** (Backpropagation Through Time),
que no es más que backprop sobre la red *desenrollada* en el tiempo.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. La recurrencia a mano en numpy

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)

# h_t = tanh(W_x·x_t + W_h·h_{t-1} + b) — misma celda en cada paso
T, n_x, n_h = 5, 3, 4
Wx = rng.standard_normal((n_h, n_x)) * 0.1
Wh = rng.standard_normal((n_h, n_h)) * 0.1
b = np.zeros(n_h)
xs = rng.standard_normal((T, n_x))

h = np.zeros(n_h)
H = []
for t in range(T):
    h = np.tanh(Wx @ xs[t] + Wh @ h + b)   # el estado se realimenta
    H.append(h)
H = np.stack(H)
print("estados h_t:", H.shape)             # (T, n_h)
print("h_final:", np.round(H[-1], 3))

## 2. `SimpleRNN` en Keras sobre una serie sinusoidal

In [ ]:
def serie(n, seed=42):
    r = np.random.default_rng(seed)
    t = np.arange(n)
    return (np.sin(0.01 * t) + r.normal(0, 0.05, n)).astype("float32")

s = serie(5000)
T = 50
X = np.stack([s[i:i + T] for i in range(len(s) - T - 1)])[..., None]  # (N, 50, 1)
y = s[T:len(s) - 1]                                                   # paso siguiente
n_tr = 4000
X_tr, y_tr, X_te, y_te = X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:]

modelo = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.SimpleRNN(20),
    layers.Dense(1),
])
modelo.compile(optimizer="adam", loss="mae")
modelo.summary()

## 3. `return_sequences`: un output al final vs uno por timestep

In [ ]:
seq_in = keras.Input(shape=(T, 1))
salida_last = layers.SimpleRNN(20, return_sequences=False)(seq_in)  # (batch, 20)
salida_seq = layers.SimpleRNN(20, return_sequences=True)(seq_in)    # (batch, T, 20)
print("return_sequences=False ->", salida_last.shape)
print("return_sequences=True  ->", salida_seq.shape)

## 4. Apilar RNNs: `return_sequences=True` en todas menos la última

In [ ]:
apilada = keras.Sequential([
    keras.Input(shape=(T, 1)),
    layers.SimpleRNN(20, return_sequences=True),  # devuelve toda la secuencia
    layers.SimpleRNN(20),                          # solo el último estado
    layers.Dense(1),
])
apilada.summary()

## 5. BPTT: el gradiente como producto de Jacobianos (vanishing/exploding)

In [ ]:
# Desenrollada, dh_T/dh_0 es el producto de los Jacobianos de cada paso.
# Con |W_h| < 1 ese producto tiende a 0 (vanishing); con |W_h| > 1 explota.
def norma_gradiente(pasos, escala):
    Wh_local = np.eye(n_h) * escala
    grad = np.eye(n_h)
    h_local = np.zeros(n_h)
    for _ in range(pasos):
        J = np.diag(1 - np.tanh(h_local) ** 2) @ Wh_local  # d/dh de tanh(W_h h)
        grad = J @ grad
    return np.linalg.norm(grad)

for esc in (0.5, 1.0, 1.2):
    print(f"|W_h|~{esc}: ||dh_50/dh_0|| = {norma_gradiente(50, esc):.2e}")

## 6. Predicción de N pasos: realimentar la propia predicción

In [ ]:
# BPTT truncado se controla con la longitud de la ventana (T).
# Predicción multi-step recursiva: el error se ACUMULA paso a paso.
ventana = X_te[0].copy()             # (T, 1)
pred = []
for _ in range(20):
    yhat = modelo.predict(ventana[None], verbose=0)[0, 0]
    pred.append(yhat)
    ventana = np.roll(ventana, -1, axis=0)
    ventana[-1, 0] = yhat            # se realimenta la predicción anterior
print("primeros 5 pasos autoregresivos:", np.round(pred[:5], 3))

## Ejercicios

1. **SimpleRNN básico**: entrená `SimpleRNN(20) → Dense(1)` sobre `sin(t)` y graficá
   predicción vs realidad; verificá MAE < 0.1.
2. **`return_sequences=True`**: apilá 2 RNN (primera con `return_sequences=True`) y
   comprobá las shapes intermedias.
3. **N pasos adelante**: extendé el bucle autoregresivo a 100 pasos y observá cómo
   diverge por acumulación de error.
4. **BPTT truncado**: con una secuencia de 200 pasos, compará entrenar con ventana
   completa vs truncada a 20 y medí el efecto en el vanishing.

## Conclusiones

- Una RNN aplica **la misma celda** en cada paso; `h_t` es su **memoria**.
- `return_sequences=True` devuelve `(batch, T, units)`; se usa para apilar o aplicar
  `Dense` por timestep. `return_state=True` expone el estado final (encoder-decoder).
- **BPTT** es backprop sobre la red desenrollada: el gradiente es un producto de
  Jacobianos → **vanishing** (`|W_h|<1`) o **exploding** (`|W_h|>1`).
- La predicción recursiva **acumula error**; el BPTT truncado limita costo y vanishing.
- Para secuencias largas (>50 pasos) conviene **LSTM/GRU** (clase 137) o Transformers.